In [ ]:
#!/usr/bin/env python3

"""
AIMES with J-Lens feedback.

Runs ONLY:
    aimes_j_lens

Controller:
    z = W_U norm(J_l h)

    c_k = sigmoid(
        standardized probe logit
    )

    aligned_c_k =
        c_k       if g_k = +1
        1-c_k     if g_k = -1

    alpha_k = 1 - aligned_c_k

    h' = h + gamma * sum_k alpha_k g_k u_k

For the final layer:
    J_l = I

Observer reads the PRE-INTERVENTION hidden state.

Expected:
    100 prompts
    x 3 objectives
    x 10 layers
    = 3000 generations/model

HF:
artifacts/multivalue_steering/<model>/v1/
generations/aimes_j_lens/
"""

 
# 0. INSTALL
 

!pip install -q -U \
    transformers \
    accelerate \
    huggingface_hub \
    safetensors \
    "pandas==2.2.3"

import sys
import gc
import json
import random
import hashlib
import shutil
import subprocess

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
)

from huggingface_hub import (
    HfApi,
    hf_hub_download,
    CommitOperationAdd,
)

from huggingface_hub.errors import (
    RemoteEntryNotFoundError,
)

from safetensors.torch import load_file

from google.colab import userdata, drive


 
# 1. MODEL SELECTION
 

MODEL_KEY = "gemma-4b" 
MODEL_KEY = "qwen-4b" 
MODEL_KEY = "llama-8b" 
MODEL_KEY = "gemma-12b" 
MODEL_KEY = "qwen-14b"


 
# 2. GLOBAL CONFIG
 

HF_REPO_ID = ""

VERSION = "v1"

METHOD = "aimes_j_lens"


HF_TOKEN = userdata.get(
    "HF_TOKEN"
)


assert HF_TOKEN is not None


hf_api = HfApi(
    token=HF_TOKEN
)


MODEL_CONFIGS = {

    "gemma-4b": {

        "model_id":
            "google/gemma-3-4b-it",

        "save_name":
            "gemma-3-4b-it",

        "display_name":
            "Gemma-3-4B-IT",

        "family":
            "gemma3",

        "num_layers":
            34,

        "layers":
            [3, 7, 10, 14, 17, 20, 24, 27, 31, 34],

        "jlens_repo":
            "neuronpedia/jacobian-lens",

        "jlens_file":
            (
                "gemma-3-4b-it/"
                "jlens/Salesforce-wikitext/"
                "gemma-3-4b-it_jacobian_lens.pt"
            ),
    },

    "qwen-4b": {

        "model_id":
            "Qwen/Qwen3-4B",

        "save_name":
            "qwen3-4b",

        "display_name":
            "Qwen3-4B",

        "family":
            "qwen3",

        "num_layers":
            36,

        "layers":
            [4, 7, 11, 14, 18, 22, 25, 29, 32, 36],

        "jlens_repo":
            "neuronpedia/jacobian-lens",

        "jlens_file":
            (
                "qwen3-4b/"
                "jlens/Salesforce-wikitext/"
                "Qwen3-4B_jacobian_lens.pt"
            ),
    },

    "llama-8b": {

        "model_id":
            "meta-llama/Llama-3.1-8B-Instruct",

        "save_name":
            "llama-3.1-8b-instruct",

        "display_name":
            "Llama-3.1-8B-Instruct",

        "family":
            "llama",

        "num_layers":
            32,

        "layers":
            [3, 6, 10, 13, 16, 19, 22, 26, 29, 32],

        "jlens_repo":
            "neuronpedia/jacobian-lens",

        "jlens_file":
            (
                "llama3.1-8b-it/"
                "jlens/Salesforce-wikitext/"
                "Llama-3.1-8B-Instruct_jacobian_lens.pt"
            ),
    },

    "gemma-12b": {

        "model_id":
            "google/gemma-3-12b-it",

        "save_name":
            "gemma-3-12b-it",

        "display_name":
            "Gemma-3-12B-IT",

        "family":
            "gemma3",

        "num_layers":
            48,

        "layers":
            [5, 10, 14, 19, 24, 29, 34, 38, 43, 48],

        "jlens_repo":
            "neuronpedia/jacobian-lens",

        "jlens_file":
            (
                "gemma-3-12b-it/"
                "jlens/Salesforce-wikitext/"
                "gemma-3-12b-it_jacobian_lens.pt"
            ),
    },

    "qwen-14b": {

        "model_id":
            "Qwen/Qwen3-14B",

        "save_name":
            "qwen3-14b",

        "display_name":
            "Qwen3-14B",

        "family":
            "qwen3",

        "num_layers":
            40,

        "layers":
            [4, 8, 12, 16, 20, 24, 28, 32, 36, 40],

        "jlens_repo":
            "neuronpedia/jacobian-lens",

        "jlens_file":
            (
                "qwen3-14b/"
                "jlens/Salesforce-wikitext/"
                "Qwen3-14B_jacobian_lens.pt"
            ),
    },
}


CFG = MODEL_CONFIGS[
    MODEL_KEY
]


MODEL_ID = CFG[
    "model_id"
]


MODEL_SAVE_NAME = CFG[
    "save_name"
]


MODEL_FAMILY = CFG[
    "family"
]


NUM_LAYERS_EXPECTED = CFG[
    "num_layers"
]


INTERVENTION_LAYERS = CFG[
    "layers"
]


 
# 3. EXPERIMENT CONFIG
 

FOUNDATIONS = [
    "Care",
    "Fairness",
    "Loyalty",
    "Authority",
    "Sanctity",
]


FOUNDATION_TO_INDEX = {

    value:
        i

    for i, value
    in enumerate(
        FOUNDATIONS
    )
}


OBJECTIVES = {

    "care_fairness": {
        "Care": +1,
        "Fairness": +1,
    },

    "loyalty_authority": {
        "Loyalty": +1,
        "Authority": +1,
    },

    "care_fairness_sanctity": {
        "Care": +1,
        "Fairness": +1,
        "Sanctity": -1,
    },
}

# add tokens as per selection/requirement
VALUE_TOKEN_SURFACES = {

    "Care":
        [" care"..................],

    "Fairness":
        [" fairness"..............],

    "Loyalty":
        [" loyalty"..............],

    "Authority":
        [" authority"............],

    "Sanctity":
        [" purity"...............],
}


N_PROMPTS_PER_FOUNDATION = 20

PROMPT_SAMPLE_SEED = 42

GAMMA = 1.0

MAX_NEW_TOKENS = 128

DO_SAMPLE = False

REPETITION_PENALTY = 1.0

GENERATION_SEED = 1234

SAVE_EVERY_N = 25

RESUME = True


 
# 4. HF PATHS
 

HF_EVAL_PROMPTS = (
    "data/evaluation/"
    "mfrc_openloop_prompts_v1.csv"
)


HF_MANIFEST_ROOT = (
    "artifacts/multivalue_steering/"
    f"manifests/{VERSION}"
)


HF_PROMPT_MANIFEST = (
    f"{HF_MANIFEST_ROOT}/"
    "multivalue_prompt_manifest.csv"
)


HF_OBJECTIVES = (
    f"{HF_MANIFEST_ROOT}/"
    "multivalue_objectives.json"
)


HF_DIRECTION_FILE = (

    f"artifacts/value_directions/"
    f"{MODEL_SAVE_NAME}/{VERSION}/directions/"
    f"{MODEL_SAVE_NAME}_"
    f"mft_value_directions_{VERSION}.safetensors"
)


HF_GENERATION_ROOT = (

    f"artifacts/multivalue_steering/"
    f"{MODEL_SAVE_NAME}/{VERSION}/generations/"
    f"{METHOD}"
)


 
# 5. DRIVE
 

if not Path(
    "/content/drive/MyDrive"
).exists():

    drive.mount(
        "/content/drive"
    )


DRIVE_ROOT = Path(

    "/content/drive/MyDrive/"
    "AIMES/multivalue_steering/"
    f"{MODEL_SAVE_NAME}/{VERSION}/generations/"
    f"{METHOD}"
)


DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


LOCAL_ROOT = Path(

    f"/content/aimes_multivalue_"
    f"{MODEL_SAVE_NAME}_{METHOD}"
)


LOCAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


CHECKPOINT_CSV = (
    DRIVE_ROOT
    /
    "generations_checkpoint.csv"
)


CHECKPOINT_TRAJECTORIES = (
    DRIVE_ROOT
    /
    "controller_trajectories_checkpoint.jsonl"
)


 
# 6. J-LENS INSTALL
 

JLENS_CODE_ROOT = Path(
    "/content/jacobian-lens"
)


if not JLENS_CODE_ROOT.exists():

    subprocess.run(

        [
            "git",
            "clone",
            "https://github.com/anthropics/"
            "jacobian-lens.git",
            str(
                JLENS_CODE_ROOT
            ),
        ],

        check=True,
    )


subprocess.run(

    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(
            JLENS_CODE_ROOT
        ),
    ],

    check=True,
)


sys.path.insert(
    0,
    str(
        JLENS_CODE_ROOT
    ),
)


import jlens


 
# 7. HELPERS
 

def clear_memory():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


def get_dtype():

    if not torch.cuda.is_available():

        return torch.float32

    if torch.cuda.is_bf16_supported():

        return torch.bfloat16

    return torch.float16


def get_input_device(
    model,
):

    for getter in [

        lambda:
            model.get_input_embeddings().weight.device,

        lambda:
            model.language_model
            .get_input_embeddings()
            .weight.device,

        lambda:
            model.model.language_model
            .get_input_embeddings()
            .weight.device,
    ]:

        try:

            return getter()

        except Exception:

            pass


    raise RuntimeError(
        "Could not determine input device."
    )


def get_transformer_layers(
    model,
):

    for getter in [

        lambda:
            model.model.layers,

        lambda:
            model.model.language_model.layers,

        lambda:
            model.language_model.layers,

        lambda:
            model.language_model.model.layers,
    ]:

        try:

            layers = getter()

            if len(
                layers
            ) == NUM_LAYERS_EXPECTED:

                return layers

        except Exception:

            pass


    raise RuntimeError(
        "Could not locate transformer layers."
    )


def condition_id(
    prompt_id,
    objective_id,
    layer,
):

    payload = "|".join(

        [

            MODEL_SAVE_NAME,

            str(
                prompt_id
            ),

            objective_id,

            str(
                layer
            ),

            METHOD,

            str(
                GAMMA
            ),
        ]
    )


    return hashlib.sha256(

        payload.encode(
            "utf-8"
        )

    ).hexdigest()


def write_json(
    path,
    obj,
):

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False,
        )


def append_jsonl(
    path,
    obj,
):

    with open(
        path,
        "a",
        encoding="utf-8",
    ) as f:

        f.write(

            json.dumps(
                obj,
                ensure_ascii=False,
            )

            +

            "\n"
        )


 
# 8. MANIFEST
 

def load_or_create_manifest():

    try:

        local = hf_hub_download(

            repo_id=
                HF_REPO_ID,

            filename=
                HF_PROMPT_MANIFEST,

            repo_type=
                "model",

            token=
                HF_TOKEN,
        )


        return pd.read_csv(
            local
        )


    except RemoteEntryNotFoundError:

        pass


    source = hf_hub_download(

        repo_id=
            HF_REPO_ID,

        filename=
            HF_EVAL_PROMPTS,

        repo_type=
            "model",

        token=
            HF_TOKEN,
    )


    source_df = pd.read_csv(
        source
    )


    parts = []


    for foundation in FOUNDATIONS:

        sub = source_df[

            source_df[
                "foundation"
            ]
            ==
            foundation
        ]


        parts.append(

            sub.sample(

                n=
                    N_PROMPTS_PER_FOUNDATION,

                random_state=
                    (
                        PROMPT_SAMPLE_SEED
                        +
                        FOUNDATION_TO_INDEX[
                            foundation
                        ]
                    ),
            )
        )


    manifest = pd.concat(

        parts,

        ignore_index=True,
    )[
        [
            "prompt_id",
            "foundation",
            "prompt",
        ]
    ]


    manifest = (

        manifest

        .sort_values(
            [
                "foundation",
                "prompt_id",
            ]
        )

        .reset_index(
            drop=True
        )
    )


    local_manifest = (
        LOCAL_ROOT
        /
        "manifest.csv"
    )


    local_objectives = (
        LOCAL_ROOT
        /
        "objectives.json"
    )


    manifest.to_csv(
        local_manifest,
        index=False,
    )


    write_json(
        local_objectives,
        OBJECTIVES,
    )


    try:

        hf_api.create_commit(

            repo_id=
                HF_REPO_ID,

            repo_type=
                "model",

            operations=[

                CommitOperationAdd(

                    path_in_repo=
                        HF_PROMPT_MANIFEST,

                    path_or_fileobj=
                        str(
                            local_manifest
                        ),
                ),

                CommitOperationAdd(

                    path_in_repo=
                        HF_OBJECTIVES,

                    path_or_fileobj=
                        str(
                            local_objectives
                        ),
                ),
            ],

            commit_message=
                "Add AIMES multi-value shared manifest",

            token=
                HF_TOKEN,
        )


    except Exception as exc:

        try:

            local = hf_hub_download(

                repo_id=
                    HF_REPO_ID,

                filename=
                    HF_PROMPT_MANIFEST,

                repo_type=
                    "model",

                token=
                    HF_TOKEN,

                force_download=
                    True,
            )


            return pd.read_csv(
                local
            )


        except Exception:

            raise exc


    return manifest


manifest_df = load_or_create_manifest()


assert len(
    manifest_df
) == (
    5
    *
    N_PROMPTS_PER_FOUNDATION
)


 
# 9. MODEL
 

dtype = get_dtype()


if MODEL_FAMILY == "gemma3":

    processor = AutoProcessor.from_pretrained(

        MODEL_ID,

        token=
            HF_TOKEN,
    )


    tokenizer = processor.tokenizer


    model = (

        Gemma3ForConditionalGeneration

        .from_pretrained(

            MODEL_ID,

            token=
                HF_TOKEN,

            dtype=
                dtype,

            device_map=
                "auto",

            low_cpu_mem_usage=
                True,
        )

        .eval()
    )


else:

    processor = None


    tokenizer = AutoTokenizer.from_pretrained(

        MODEL_ID,

        token=
            HF_TOKEN,

        trust_remote_code=
            True,
    )


    model = (

        AutoModelForCausalLM

        .from_pretrained(

            MODEL_ID,

            token=
                HF_TOKEN,

            dtype=
                dtype,

            device_map=
                "auto",

            low_cpu_mem_usage=
                True,

            trust_remote_code=
                True,
        )

        .eval()
    )


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


INPUT_DEVICE = get_input_device(
    model
)


TRANSFORMER_LAYERS = get_transformer_layers(
    model
)


 
# 10. DIRECTIONS
 

direction_local = hf_hub_download(

    repo_id=
        HF_REPO_ID,

    filename=
        HF_DIRECTION_FILE,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


UNIT_DIRECTIONS = (

    load_file(
        direction_local
    )[
        "unit_directions"
    ]

    .float()

    .cpu()
)


assert UNIT_DIRECTIONS.shape[
    1
] == NUM_LAYERS_EXPECTED


 
# 11. J-LENS
 

lens_model = jlens.from_hf(

    model,

    tokenizer,

    force_bos=True,
)


lens_file = hf_hub_download(

    repo_id=
        CFG[
            "jlens_repo"
        ],

    filename=
        CFG[
            "jlens_file"
        ],

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


jacobian_lens = (

    jlens
    .JacobianLens
    .load(
        lens_file
    )
)


J_MATRICES = {}


for layer in INTERVENTION_LAYERS:

    layer_idx = (
        layer - 1
    )


    if layer == NUM_LAYERS_EXPECTED:

        continue


    J_MATRICES[
        layer_idx
    ] = jacobian_lens.jacobians[
        layer_idx
    ]


 
# 12. PROBES
 

TOKEN_IDS = {}


for foundation in FOUNDATIONS:

    ids = []


    for surface in VALUE_TOKEN_SURFACES[
        foundation
    ]:

        encoded = tokenizer.encode(

            surface,

            add_special_tokens=False,
        )


        if len(
            encoded
        ) != 1:

            raise RuntimeError(

                f"{foundation}: "
                f"{surface!r} -> {encoded}"
            )


        ids.append(
            encoded[
                0
            ]
        )


    TOKEN_IDS[
        foundation
    ] = ids


 
# 13. PROMPT
 

def render_prompt(
    prompt,
):

    if MODEL_FAMILY == "gemma3":

        return processor.apply_chat_template(

            [

                {
                    "role":
                        "user",

                    "content": [

                        {
                            "type":
                                "text",

                            "text":
                                str(
                                    prompt
                                ),
                        }
                    ],
                }
            ],

            tokenize=False,

            add_generation_prompt=True,
        )


    kwargs = {

        "tokenize":
            False,

        "add_generation_prompt":
            True,
    }


    if MODEL_FAMILY == "qwen3":

        kwargs[
            "enable_thinking"
        ] = False


    messages = [

        {
            "role":
                "user",

            "content":
                str(
                    prompt
                ),
        }
    ]


    try:

        return tokenizer.apply_chat_template(

            messages,

            **kwargs,
        )


    except TypeError:

        kwargs.pop(
            "enable_thinking",
            None,
        )


        return tokenizer.apply_chat_template(

            messages,

            **kwargs,
        )


def encode_prompt(
    prompt,
):

    encoded = tokenizer(

        render_prompt(
            prompt
        ),

        return_tensors=
            "pt",

        add_special_tokens=
            False,
    )


    return {

        key:
            value.to(
                INPUT_DEVICE
            )

        for key, value
        in encoded.items()

        if torch.is_tensor(
            value
        )
    }


 
# 14. J-LENS OBSERVER
 

@torch.inference_mode()
def j_lens_confidences(
    hidden_last,
    layer_idx,
):

    residual = hidden_last.float()


    # Final layer uses identity.
    if layer_idx != (
        NUM_LAYERS_EXPECTED
        -
        1
    ):

        J = J_MATRICES[
            layer_idx
        ].to(

            residual.device,

            dtype=
                residual.dtype,
        )


        residual = (
            residual
            @
            J.T
        )


    logits = (

        lens_model

        .unembed(
            residual
        )

        .float()
    )


    vocab_mean = logits.mean(
        dim=-1
    )


    vocab_std = logits.std(

        dim=-1,

        correction=0,

    ).clamp_min(
        1e-6
    )


    result = {}


    for value in FOUNDATIONS:

        selected = logits[
            :,
            TOKEN_IDS[
                value
            ]
        ]


        standardized = (

            selected

            -

            vocab_mean[
                :,
                None
            ]

        ) / vocab_std[
            :,
            None
        ]


        score = standardized.mean(
            dim=-1
        )


        result[
            value
        ] = float(

            torch.sigmoid(
                score
            )[
                0
            ]

            .detach()

            .cpu()
        )


    return result


 
# 15. AIMES-JL CONTROLLER
 

class AIMESJLController:

    def __init__(
        self,
        objective,
        layer,
    ):

        self.objective = objective

        self.layer = layer

        self.layer_idx = (
            layer - 1
        )

        self.step = 0

        self.trajectory = []


    def __call__(
        self,
        module,
        inputs,
        output,
    ):

        if torch.is_tensor(
            output
        ):

            hidden = output

            output_type = "tensor"


        elif isinstance(
            output,
            tuple,
        ):

            hidden = output[
                0
            ]

            output_type = "tuple"


        else:

            raise RuntimeError(
                "Unexpected output."
            )


        modified = hidden.clone()


        # PRE-INTERVENTION
        h = hidden[
            :,
            -1,
            :
        ]


        confidences = j_lens_confidences(

            h,

            self.layer_idx,
        )


        alphas = {}


        for value, sign in self.objective.items():

            c = confidences[
                value
            ]


            aligned_c = (

                c

                if sign == +1

                else

                1.0 - c
            )


            alphas[
                value
            ] = (

                1.0
                -
                aligned_c
            )


        steering = torch.zeros_like(
            h
        )


        for value, sign in self.objective.items():

            direction = UNIT_DIRECTIONS[

                FOUNDATION_TO_INDEX[
                    value
                ],

                self.layer_idx,

            ].to(

                h.device,

                dtype=
                    h.dtype,
            )


            steering += (

                float(
                    alphas[
                        value
                    ]
                )

                *

                float(
                    sign
                )

                *

                direction[
                    None,
                    :
                ]
            )


        modified[
            :,
            -1,
            :
        ] += (

            GAMMA
            *
            steering
        )


        self.trajectory.append(

            {

                "step":
                    self.step,

                "layer":
                    self.layer,

                "confidence":
                    {
                        value:
                            float(
                                confidences[
                                    value
                                ]
                            )

                        for value
                        in self.objective
                    },

                "alpha":
                    {
                        value:
                            float(
                                alphas[
                                    value
                                ]
                            )

                        for value
                        in self.objective
                    },
            }
        )


        self.step += 1


        if output_type == "tensor":

            return modified


        return (

            modified,

            *output[
                1:
            ],
        )


 
# 16. GENERATION
 

def seed_all(
    seed,
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


@torch.inference_mode()
def baseline_generate(
    prompt,
    seed,
):

    seed_all(
        seed
    )


    inputs = encode_prompt(
        prompt
    )


    prompt_len = inputs[
        "input_ids"
    ].shape[
        1
    ]


    output = model.generate(

        **inputs,

        max_new_tokens=
            MAX_NEW_TOKENS,

        do_sample=
            DO_SAMPLE,

        repetition_penalty=
            REPETITION_PENALTY,

        pad_token_id=
            tokenizer.pad_token_id,

        eos_token_id=
            tokenizer.eos_token_id,

        use_cache=
            True,
    )


    text = tokenizer.decode(

        output[
            0,
            prompt_len:
        ],

        skip_special_tokens=
            True,
    ).strip()


    del inputs
    del output

    clear_memory()


    return text


@torch.inference_mode()
def steered_generate(
    prompt,
    seed,
    objective,
    layer,
):

    seed_all(
        seed
    )


    inputs = encode_prompt(
        prompt
    )


    prompt_len = inputs[
        "input_ids"
    ].shape[
        1
    ]


    controller = AIMESJLController(

        objective,

        layer,
    )


    handle = TRANSFORMER_LAYERS[

        layer - 1

    ].register_forward_hook(
        controller
    )


    try:

        output = model.generate(

            **inputs,

            max_new_tokens=
                MAX_NEW_TOKENS,

            do_sample=
                DO_SAMPLE,

            repetition_penalty=
                REPETITION_PENALTY,

            pad_token_id=
                tokenizer.pad_token_id,

            eos_token_id=
                tokenizer.eos_token_id,

            use_cache=
                True,
        )


    finally:

        handle.remove()


    text = tokenizer.decode(

        output[
            0,
            prompt_len:
        ],

        skip_special_tokens=
            True,
    ).strip()


    trajectory = (
        controller.trajectory
    )


    del inputs
    del output

    clear_memory()


    return (
        text,
        trajectory,
    )


 
# 17. RESUME
 

rows = []


if (
    RESUME
    and
    CHECKPOINT_CSV.exists()
):

    rows = (

        pd.read_csv(
            CHECKPOINT_CSV
        )

        .to_dict(
            orient="records"
        )
    )


completed = {

    str(
        row[
            "condition_id"
        ]
    )

    for row in rows
}


baseline_cache = {

    str(
        row[
            "prompt_id"
        ]
    ):
        str(
            row[
                "baseline_response"
            ]
        )

    for row in rows
}


 
# 18. LOOP
 

new_since_save = 0


for prompt_index, prompt_row in manifest_df.iterrows():

    prompt_id = str(
        prompt_row[
            "prompt_id"
        ]
    )


    prompt = str(
        prompt_row[
            "prompt"
        ]
    )


    foundation = str(
        prompt_row[
            "foundation"
        ]
    )


    seed = (
        GENERATION_SEED
        +
        prompt_index
    )


    if prompt_id not in baseline_cache:

        baseline_cache[
            prompt_id
        ] = baseline_generate(

            prompt,

            seed,
        )


    baseline = baseline_cache[
        prompt_id
    ]


    for objective_id, objective in OBJECTIVES.items():

        for layer in INTERVENTION_LAYERS:

            cid = condition_id(

                prompt_id,

                objective_id,

                layer,
            )


            if cid in completed:

                continue


            print(

                f"\r{MODEL_SAVE_NAME} | "
                f"{METHOD} | "
                f"{prompt_index+1}/{len(manifest_df)} | "
                f"{objective_id} | L{layer}",

                end="",
            )


            response, trajectory = steered_generate(

                prompt,

                seed,

                objective,

                layer,
            )


            row = {

                "condition_id":
                    cid,

                "model_key":
                    MODEL_KEY,

                "model_id":
                    MODEL_ID,

                "model_save_name":
                    MODEL_SAVE_NAME,

                "version":
                    VERSION,

                "method":
                    METHOD,

                "prompt_id":
                    prompt_id,

                "prompt_foundation":
                    foundation,

                "prompt":
                    prompt,

                "objective_id":
                    objective_id,

                "requested_values":
                    json.dumps(
                        list(
                            objective.keys()
                        )
                    ),

                "requested_signs":
                    json.dumps(
                        list(
                            objective.values()
                        )
                    ),

                "objective_json":
                    json.dumps(
                        objective,
                        sort_keys=True,
                    ),

                "layer":
                    layer,

                "num_layers":
                    NUM_LAYERS_EXPECTED,

                "relative_depth":
                    layer
                    /
                    NUM_LAYERS_EXPECTED,

                "gamma":
                    GAMMA,

                "baseline_response":
                    baseline,

                "steered_response":
                    response,

                "response_changed":
                    int(
                        response
                        !=
                        baseline
                    ),

                "generation_seed":
                    seed,

                "max_new_tokens":
                    MAX_NEW_TOKENS,

                "do_sample":
                    DO_SAMPLE,

                "repetition_penalty":
                    REPETITION_PENALTY,

                "controller_steps":
                    len(
                        trajectory
                    ),

                "created_at":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            }


            for value in FOUNDATIONS:

                alpha_values = [

                    step[
                        "alpha"
                    ][
                        value
                    ]

                    for step
                    in trajectory

                    if value
                    in step[
                        "alpha"
                    ]
                ]


                if alpha_values:

                    row[
                        f"mean_alpha_{value.lower()}"
                    ] = float(
                        np.mean(
                            alpha_values
                        )
                    )


                    row[
                        f"final_alpha_{value.lower()}"
                    ] = float(
                        alpha_values[
                            -1
                        ]
                    )


                    row[
                        f"min_alpha_{value.lower()}"
                    ] = float(
                        np.min(
                            alpha_values
                        )
                    )


                    row[
                        f"max_alpha_{value.lower()}"
                    ] = float(
                        np.max(
                            alpha_values
                        )
                    )


                else:

                    for stat in [
                        "mean",
                        "final",
                        "min",
                        "max",
                    ]:

                        row[
                            f"{stat}_alpha_"
                            f"{value.lower()}"
                        ] = np.nan


            rows.append(
                row
            )


            completed.add(
                cid
            )


            append_jsonl(

                CHECKPOINT_TRAJECTORIES,

                {

                    "condition_id":
                        cid,

                    "method":
                        METHOD,

                    "trajectory":
                        trajectory,
                },
            )


            new_since_save += 1


            if (
                new_since_save
                >=
                SAVE_EVERY_N
            ):

                pd.DataFrame(
                    rows
                ).to_csv(

                    CHECKPOINT_CSV,

                    index=False,
                )


                print(
                    f"\nCheckpoint: {len(rows)}"
                )


                new_since_save = 0


 
# 19. VALIDATE
 

df = pd.DataFrame(
    rows
)


expected = (

    len(
        manifest_df
    )

    *
    len(
        OBJECTIVES
    )

    *
    len(
        INTERVENTION_LAYERS
    )
)


print(
    "\nExpected:",
    expected
)


print(
    "Observed:",
    len(
        df
    )
)


if len(
    df
) != expected:

    raise RuntimeError(
        "Incomplete AIMES-JL generation."
    )


if df[
    "condition_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate condition IDs."
    )


 
# 20. SAVE
 

generation_csv = (
    LOCAL_ROOT
    /
    "multivalue_generations.csv"
)


generation_jsonl = (
    LOCAL_ROOT
    /
    "multivalue_generations.jsonl"
)


controller_summary_csv = (
    LOCAL_ROOT
    /
    "controller_summary.csv"
)


trajectory_jsonl = (
    LOCAL_ROOT
    /
    "controller_trajectories.jsonl"
)


metadata_json = (
    LOCAL_ROOT
    /
    "generation_metadata.json"
)


df.to_csv(
    generation_csv,
    index=False,
)


df.to_json(

    generation_jsonl,

    orient="records",

    lines=True,

    force_ascii=False,
)


alpha_cols = [

    col

    for col
    in df.columns

    if "alpha_" in col
]


df[

    [

        "condition_id",
        "model_save_name",
        "prompt_id",
        "objective_id",
        "method",
        "layer",
        "relative_depth",
        "controller_steps",
    ]

    +

    alpha_cols

].to_csv(

    controller_summary_csv,

    index=False,
)


if CHECKPOINT_TRAJECTORIES.exists():

    shutil.copy2(

        CHECKPOINT_TRAJECTORIES,

        trajectory_jsonl,
    )


write_json(

    metadata_json,

    {

        "experiment":
            "AIMES multi-value steering",

        "method":
            METHOD,

        "observer":
            "j_lens",

        "model_key":
            MODEL_KEY,

        "model_id":
            MODEL_ID,

        "model_save_name":
            MODEL_SAVE_NAME,

        "objectives":
            OBJECTIVES,

        "layers":
            INTERVENTION_LAYERS,

        "gamma":
            GAMMA,

        "expected_rows":
            expected,

        "value_token_surfaces":
            VALUE_TOKEN_SURFACES,

        "j_lens_file":
            CFG[
                "jlens_file"
            ],
    },
)


 
# 21. ONE HF COMMIT
 

operations = []


for path in [

    generation_csv,
    generation_jsonl,
    controller_summary_csv,
    trajectory_jsonl,
    metadata_json,

]:

    if Path(
        path
    ).exists():

        operations.append(

            CommitOperationAdd(

                path_in_repo=
                    f"{HF_GENERATION_ROOT}/"
                    f"{Path(path).name}",

                path_or_fileobj=
                    str(
                        path
                    ),
            )
        )


commit = hf_api.create_commit(

    repo_id=
        HF_REPO_ID,

    repo_type=
        "model",

    operations=
        operations,

    commit_message=
        (
            "Add AIMES-JL multi-value generations "
            f"for {MODEL_SAVE_NAME}"
        ),

    token=
        HF_TOKEN,
)


print(
    "\nAIMES-JL COMPLETE"
)


print(
    "Rows:",
    len(
        df
    )
)


print(
    "HF:",
    HF_GENERATION_ROOT
)


print(
    commit
)